# Red-Blue Chess Engine Tournament

**Scenario 2 — Zero-Sum Adversarial Evaluation**

A **Red (attacker)** agent probes each engine for failures across 20 curated positions spanning five categories: opening, tactical, material, endgame, and edge cases.
A **Blue (defender)** agent cross-validates engine responses by majority consensus.

| Failure type | Points to Red |
|---|---|
| Illegal move | 3 |
| Crash / None | 3 |
| Timeout | 2 |
| Blunder (mate-in-1 missed) | 2 |
| Eval out of range | 1 |

An engine's **defender score** = total positions − attacker points conceded.

In [ ]:
# Install dependencies — safe to re-run, skips already-installed packages
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'chess', 'pandas', 'matplotlib', 'numpy'])
print('Dependencies ready.')

In [ ]:
import sys, os

# Hardcoded absolute paths — reliable regardless of where Jupyter was launched
_repo      = os.path.expanduser('~/CubistChessEngine')
_scenarios = os.path.join(_repo, 'scenarios')

for _p in [_repo, _scenarios]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

assert os.path.isfile(os.path.join(_scenarios, 'red_blue.py')), \
    f'red_blue.py not found at {_scenarios} — check that REPO_ROOT is correct'

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import display

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#ccc',
    'ytick.color':      '#ccc',
    'text.color':       '#eee',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'axes.titlecolor':  '#fff',
    'font.family':      'monospace',
})

RED   = '#e05252'
BLUE  = '#5b8cde'
GREEN = '#5cb85c'
AMBER = '#e8a838'
GREY  = '#888888'

print('Setup complete.')

## Run the Tournament

This cell imports the Red-Blue module and runs a full tournament against all available engines.
Engines that fail to import are skipped automatically.

In [ ]:
from red_blue import (
    POSITIONS, FAILURE_POINTS,
    build_engines,
    RedAgent, BlueAgent, Scorer, Report,
    ProbeResult, EngineScore, ConsensusResult,
)

print('Loading engines...')
engines = build_engines()

red    = RedAgent()
blue   = BlueAgent()
scorer = Scorer()

all_results: list[ProbeResult] = []
all_scores:  list[EngineScore] = []

for engine in engines:
    print(f'  Probing {engine.name}...')
    results = red.run_gauntlet(engine, POSITIONS)
    all_results.extend(results)
    all_scores.append(scorer.compute(engine.name, results))

ranked = scorer.rank(all_scores)

consensus: list[ConsensusResult] = []
for pos in POSITIONS:
    c = blue.cross_validate(engines, pos)
    consensus.append(c)

print(f'\nDone. {len(engines)} engines x {len(POSITIONS)} positions = {len(all_results)} probes.')

## Overall Leaderboard

In [ ]:
total = ranked[0].total_positions if ranked else 0

rows = []
for s in ranked:
    ill = s.failures_by_type.get('illegal_move', 0) + s.failures_by_type.get('crash', 0)
    rows.append({
        'Engine':          s.engine_name,
        'Defender Score':  f"{s.defender_points}/{total}",
        'Reliability':     f"{s.reliability_pct}%",
        'Illegal / Crash': ill,
        'Blunders':        s.failures_by_type.get('blunder', 0),
        'Timeouts':        s.failures_by_type.get('timeout', 0),
        'Eval Errors':     s.failures_by_type.get('eval_wrong', 0),
        'Worst Category':  s.worst_category,
    })

df_board = pd.DataFrame(rows)

def _color_reliability(val):
    pct = float(val.strip('%'))
    if pct >= 80:
        return 'color: #5cb85c; font-weight: bold'
    if pct >= 50:
        return 'color: #e8a838'
    return 'color: #e05252'

styled = (
    df_board.style
    .set_caption('Red-Blue Tournament — Final Standings')
    .map(_color_reliability, subset=['Reliability'])
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size', '1.1em'), ('font-weight', 'bold'), ('padding', '6px')]
    }])
    .hide(axis='index')
)
display(styled)

## Reliability & Defender Score

In [ ]:
names      = [s.engine_name     for s in ranked]
reliab     = [s.reliability_pct for s in ranked]
def_scores = [s.defender_points for s in ranked]
x = np.arange(len(names))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Engine Reliability & Defender Scores', fontsize=13, fontweight='bold', y=1.02)

bar_colors = [GREEN if r >= 80 else AMBER if r >= 50 else RED for r in reliab]
bars1 = ax1.barh(x, reliab, color=bar_colors, edgecolor='#333', height=0.6)
ax1.set_yticks(x)
ax1.set_yticklabels(names)
ax1.set_xlabel('Reliability (%)')
ax1.set_title('Reliability (% positions with no failure)')
ax1.set_xlim(0, 110)
ax1.axvline(80, color=GREEN, linestyle='--', linewidth=1, alpha=0.6, label='80% threshold')
ax1.legend(fontsize=8)
ax1.grid(axis='x')
for bar, val in zip(bars1, reliab):
    ax1.text(val + 1.5, bar.get_y() + bar.get_height() / 2,
             f'{val:.0f}%', va='center', fontsize=9)

score_colors = [BLUE if d >= 0 else RED for d in def_scores]
bars2 = ax2.barh(x, def_scores, color=score_colors, edgecolor='#333', height=0.6)
ax2.set_yticks(x)
ax2.set_yticklabels(names)
ax2.set_xlabel('Defender Score')
ax2.set_title('Defender Score (positions - attacker pts)')
ax2.axvline(0, color='#666', linewidth=1)
ax2.grid(axis='x')
for bar, val in zip(bars2, def_scores):
    offset = 0.4 if val >= 0 else -0.4
    ax2.text(val + offset, bar.get_y() + bar.get_height() / 2,
             str(val), va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.show()

## Failure Breakdown by Type

In [ ]:
failure_types = ['crash', 'illegal_move', 'timeout', 'blunder', 'eval_wrong']
type_colors   = [RED, '#c0392b', AMBER, '#e67e22', GREY]
type_labels   = ['Crash', 'Illegal Move', 'Timeout', 'Blunder', 'Eval Error']

matrix = np.array([
    [s.failures_by_type.get(ft, 0) for ft in failure_types]
    for s in ranked
])

fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle('Failure Breakdown by Type', fontsize=13, fontweight='bold')

left = np.zeros(len(ranked))
for ft, color, label in zip(failure_types, type_colors, type_labels):
    vals = matrix[:, failure_types.index(ft)]
    ax.barh(x, vals, left=left, color=color, edgecolor='#111',
            height=0.6, label=f'{label} ({FAILURE_POINTS[ft]}pt)')
    for j, (v, l) in enumerate(zip(vals, left)):
        if v > 0:
            ax.text(l + v / 2, j, str(v), ha='center', va='center',
                    fontsize=8, color='white', fontweight='bold')
    left += vals

ax.set_yticks(x)
ax.set_yticklabels(names)
ax.set_xlabel('Number of failures')
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='x')
plt.tight_layout()
plt.show()

## Failure Heatmap by Category

In [ ]:
categories = ['opening', 'tactical', 'material', 'endgame', 'edge_case']

heat = np.array([
    [s.failures_by_category.get(cat, 0) for cat in categories]
    for s in ranked
], dtype=float)

fig, ax = plt.subplots(figsize=(9, 0.8 * len(ranked) + 1.5))
fig.suptitle('Failures per Engine per Category', fontsize=13, fontweight='bold')

im = ax.imshow(heat, cmap='Reds', aspect='auto', vmin=0)
ax.set_xticks(range(len(categories)))
ax.set_xticklabels([c.replace('_', ' ').title() for c in categories])
ax.set_yticks(range(len(ranked)))
ax.set_yticklabels(names)

for i in range(len(ranked)):
    for j in range(len(categories)):
        val = int(heat[i, j])
        text_color = 'white' if heat.max() > 0 and val >= heat.max() * 0.6 else '#ccc'
        ax.text(j, i, str(val), ha='center', va='center',
                fontsize=11, fontweight='bold', color=text_color)

plt.colorbar(im, ax=ax, label='Failures', shrink=0.8)
plt.tight_layout()
plt.show()

## Head-to-Head Matrix

Each cell shows **engine A wins − engine B wins** on shared positions. Positive = row engine won more positions.

In [ ]:
n = len(ranked)
h2h_matrix = np.zeros((n, n))
annot = np.empty((n, n), dtype=object)

for i, sa in enumerate(ranked):
    for j, sb in enumerate(ranked):
        if i == j:
            annot[i, j] = '-'
            h2h_matrix[i, j] = 0
        else:
            h = scorer.head_to_head(sa.engine_name, sb.engine_name, all_results)
            h2h_matrix[i, j] = h['wins_a'] - h['wins_b']
            annot[i, j] = f"{h['wins_a']}-{h['wins_b']}"

fig, ax = plt.subplots(figsize=(0.9 * n + 2, 0.9 * n + 2))
fig.suptitle('Head-to-Head Win Matrix (row vs col)', fontsize=13, fontweight='bold')

masked = np.ma.masked_where(np.eye(n, dtype=bool), h2h_matrix)
im = ax.imshow(masked, cmap='RdYlGn', aspect='auto',
               vmin=-len(POSITIONS), vmax=len(POSITIONS))

ax.set_xticks(range(n))
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_yticks(range(n))
ax.set_yticklabels(names)

for i in range(n):
    for j in range(n):
        color = 'white' if abs(h2h_matrix[i, j]) > len(POSITIONS) * 0.4 else '#ddd'
        if i == j:
            color = '#555'
        ax.text(j, i, annot[i, j], ha='center', va='center',
                fontsize=9, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Net wins (row - col)', shrink=0.8)
plt.tight_layout()
plt.show()

## Hardest Positions

Positions that caused the most failures across all engines.

In [ ]:
pos_fail_count: dict[str, int] = {}
pos_fail_pts:   dict[str, int] = {}
pos_category:   dict[str, str] = {}

for r in all_results:
    if r.failure_reason:
        pos_fail_count[r.position_name] = pos_fail_count.get(r.position_name, 0) + 1
        pos_fail_pts[r.position_name]   = pos_fail_pts.get(r.position_name, 0) + r.attacker_points
        pos_category[r.position_name]   = r.category

killer_positions = sorted(pos_fail_count, key=pos_fail_count.get, reverse=True)

df_killers = pd.DataFrame([
    {
        'Position':       p,
        'Category':       pos_category.get(p, ''),
        'Engines Failed': pos_fail_count[p],
        'Total Red Pts':  pos_fail_pts[p],
    }
    for p in killer_positions
])

fig, ax = plt.subplots(figsize=(11, max(4, 0.45 * len(killer_positions) + 1.5)))
fig.suptitle('Position Difficulty — Engines Failed per Position', fontsize=13, fontweight='bold')

short_names = [p[:32] + ('...' if len(p) > 32 else '') for p in killer_positions]
ypos = np.arange(len(killer_positions))
fail_vals = [pos_fail_count[p] for p in killer_positions]

cat_palette = {
    'opening': BLUE, 'tactical': RED, 'material': AMBER,
    'endgame': GREEN, 'edge_case': '#9b59b6'
}
bar_c = [cat_palette.get(pos_category.get(p, ''), GREY) for p in killer_positions]

bars = ax.barh(ypos, fail_vals, color=bar_c, edgecolor='#222', height=0.6)
ax.set_yticks(ypos)
ax.set_yticklabels(short_names, fontsize=9)
ax.set_xlabel('Number of engines that failed')
ax.set_xticks(range(0, len(engines) + 1))
ax.grid(axis='x')

for bar, val in zip(bars, fail_vals):
    ax.text(val + 0.05, bar.get_y() + bar.get_height() / 2,
            f'{val}', va='center', fontsize=9)

legend_patches = [mpatches.Patch(color=v, label=k.replace('_', ' ').title())
                  for k, v in cat_palette.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

display(df_killers.style.hide(axis='index')
        .set_caption('Hardest positions')
        .set_properties(**{'text-align': 'center'}))

## Consensus Analysis

The **Blue agent** cross-validates moves across all engines. Positions where 4+ engines agree on a move but at least one engine dissents are flagged as outlier positions.

In [ ]:
flagged = [c for c in consensus if c.outlier_flagged]

if flagged:
    rows = [{
        'Position':         c.position_name,
        'Majority Move':    c.majority_move or 'N/A',
        'Agreeing Engines': ', '.join(c.agreeing_engines),
        'Outlier Engines':  ', '.join(c.outlier_engines),
    } for c in flagged]
    df_consensus = pd.DataFrame(rows)
    display(df_consensus.style.hide(axis='index')
            .set_caption(f'Blue Agent — {len(flagged)} position(s) with outlier disagreement')
            .set_properties(**{'text-align': 'left'}))
else:
    print('No consensus outliers detected.')

agree_rate = round((1 - len(flagged) / len(POSITIONS)) * 100, 1)
print(f'Blue agent consensus rate: {agree_rate}% of positions had full or near-full agreement.')

## Per-Engine Detail

Full probe results for every engine and position.

In [ ]:
df_detail = pd.DataFrame([
    {
        'Engine':   r.engine_name,
        'Position': r.position_name,
        'Category': r.category,
        'Move':     r.move_returned or '-',
        'Eval (cp)': r.eval_returned if r.eval_returned is not None else '-',
        'Legal':    '✓' if r.is_legal else '✗',
        'Failure':  r.failure_reason or '-',
        'Red Pts':  r.attacker_points,
        'Time (s)': r.time_taken,
    }
    for r in all_results
])

def _color_failure(val):
    if val in ('illegal_move', 'crash'):  return 'color: #e05252; font-weight: bold'
    if val in ('blunder', 'timeout'):     return 'color: #e8a838'
    if val == 'eval_wrong':               return 'color: #aaa'
    return ''

def _color_pts(val):
    if val >= 3: return 'color: #e05252; font-weight: bold'
    if val >= 2: return 'color: #e8a838'
    if val == 1: return 'color: #aaa'
    return 'color: #5cb85c'

display(
    df_detail.style
    .map(_color_failure, subset=['Failure'])
    .map(_color_pts,     subset=['Red Pts'])
    .set_caption('Full Probe Results')
    .hide(axis='index')
    .set_properties(**{'text-align': 'center', 'font-size': '11px'})
)

## Save Report

Saves a timestamped Markdown report to `scenarios/red_blue_results/`.

In [ ]:
report = Report()
flagged_consensus = [c for c in consensus if c.outlier_flagged]
md_path = report.write_markdown(ranked, all_results, flagged_consensus)
print(f'Report saved → {md_path}')